# 📋 [K06] 군 공지 사항 요약 AI - 분대 공지 요약 카드 만들기
### 정답 해설지 & 모범 실습 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K06/K06_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 일반행정·부대관리 (Level 1) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 15.0시간) |
| **실습 유형** | 생성형 AI 프롬프트 설계, 가상 데이터 분석, 대조 점검 워크북 |

### 📖 프로젝트 목표
장병은 하루에도 여러 부서의 공지를 받지만, 긴 공지 속에서 기한·대상·준비물을 놓쳐 불이익을 겪는 일이 잦습니다. 이 프로젝트에서는 공개 생성형 AI로 가상 부대 공지를 요약하는 방법을 익히고, 역할 지정과 조건 제시로 프롬프트를 다듬어 '분대 공지 요약 카드'를 완성합니다. 최종적으로 누구나 다시 쓸 수 있는 나만의 공지 요약 프롬프트를 만들어, 요약 결과를 원문과 대조해 확인하는 습관까지 기릅니다.

> **⚠️ [교육용 가상 자료 원칙]**
> - 본 워크북의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> - 실제 군사 기밀이나 부대 세부 인적·작전 정보는 절대 외부 AI 도구에 입력하지 않습니다.

---

> **💡 [워크북 활용 안내]**
> - 본 실습은 파이썬 코딩 지식을 요구하지 않으며, 공개 생성형 AI(ChatGPT, Claude 등)와의 상호작용 및 팩트체크 검증을 기록하는 포트폴리오입니다.
> - 코드 셀의 따옴표(`""" ... """`) 사이에 프롬프트와 결과를 입력하고 실행(`Shift + Enter`)하면, 서식에 맞춰 시각적으로 자동 렌더링됩니다.
> - 📌 **본 파일은 모든 미션의 모범 답안과 팩트체크 내역이 수록된 '정답 해설지'입니다.**



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 공지 데이터 뷰어
# ============================================================================
from IPython.display import display, Markdown
import json
import os

# 교육용 가상 공지 데이터베이스 (N01 ~ N12, 추가 공지 A1 ~ A2)
NOTICES_DB = {
    "N01": {"date": "11-02", "dept": "대대 행정과", "title": "11월 정기 휴가 신청 접수 안내", "content": "11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. 휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. 결과는 11월 9일(월) 중대 게시판에 게시합니다. 문의: 행정과 담당관(내선 1101)."},
    "N02": {"date": "11-02", "dept": "대대 의무실", "title": "독감 예방접종 일정 안내", "content": "독감 예방접종을 11월 10일(화)부터 11월 12일(목)까지 중대별로 실시합니다. 1중대는 10일 09시, 2중대는 11일 09시, 3중대는 12일 09시에 의무실 앞으로 집합합니다. 접종 당일 발열이 있는 인원은 접종 전 의무병에게 알리고, 접종 후 30분간 의무실 대기 후 복귀합니다. 미접종자는 11월 16일(월) 추가 접종을 받습니다."},
    "N03": {"date": "11-03", "dept": "대대 군수과", "title": "동계 피복 보급 일정", "content": "동계 피복(방한 내피, 방한 장갑, 넥워머)을 11월 6일(금) 13시부터 17시까지 중대 순서대로 보급합니다. 분대장은 분대원 치수표를 11월 4일(수)까지 군수과에 제출합니다. 치수 교환은 보급 당일에만 가능하며, 이후에는 다음 달 정기 보급 때 교환합니다."},
    "N04": {"date": "11-03", "dept": "대대 정보통신실", "title": "사이버 보안 점검 및 비밀번호 변경", "content": "11월 9일(월)부터 13일(금)까지 사이버 보안 점검을 실시합니다. 모든 인원은 11월 8일(일)까지 업무용 계정 비밀번호를 변경해야 합니다. 비밀번호는 영문·숫자·특수문자를 포함해 9자리 이상으로 설정하고, 이전 비밀번호와 같으면 안 됩니다. 개인 휴대전화로 업무 자료를 촬영하거나 전송하는 행위는 금지됩니다."},
    "N05": {"date": "11-04", "dept": "대대 교육훈련과", "title": "11월 사격 훈련 일정 변경", "content": "기상 악화 예보로 11월 6일(금) 예정이던 2중대 사격 훈련을 11월 13일(금)로 연기합니다. 1중대와 3중대 일정은 변경 없습니다. 2중대는 6일에 실내 교육(사격술 예비훈련)으로 대체하며, 장소는 대대 강당입니다. 세부 시간표는 중대장 지시에 따릅니다."},
    "N06": {"date": "11-04", "dept": "대대 복지과", "title": "부대 내 복지시설 운영 시간 조정", "content": "동절기를 맞아 11월 9일(월)부터 체력단련실 운영 시간을 06시~21시로, 사이버지식정보방 운영 시간을 18시~21시 30분으로 조정합니다. 매주 수요일 14시~16시는 체력단련실 소독으로 이용할 수 없습니다. 복지매장(PX) 운영 시간은 기존과 같습니다."},
    "N07": {"date": "11-05", "dept": "대대 인사과", "title": "자기개발 지원 신청 안내", "content": "2027년 상반기 자기개발 비용 지원 신청을 11월 20일(금)까지 받습니다. 지원 대상은 자격증 응시료와 온라인 강좌 수강료이며, 1인당 연간 한도 내에서 실비를 지원합니다. 신청 시 응시 확인서 또는 수강 등록 확인서를 첨부합니다. 지원 여부는 12월 첫째 주에 개별 통보합니다."},
    "N08": {"date": "11-05", "dept": "대대 행정과", "title": "11월 정기 휴가 신청 접수 기한 연장", "content": "11월 정기 휴가 신청 접수 기한을 11월 5일(목) 18시에서 11월 6일(금) 12시로 연장합니다. 사격 훈련 일정 변경에 따른 조치입니다. 결과 게시일(11월 9일)은 변경 없습니다."},
    "N09": {"date": "11-06", "dept": "대대 안전관리관", "title": "동절기 안전사고 예방 수칙", "content": "동절기 안전사고 예방을 위해 다음 사항을 지킵니다. 생활관 내 개인 전열기 사용을 금지합니다. 차량 운행 전 타이어 상태와 결빙 구간을 확인합니다. 야외 활동 시 방한 장구를 착용하고 2인 1조로 움직입니다. 동상 의심 증상이 있으면 즉시 의무실에 알립니다."},
    "N10": {"date": "11-06", "dept": "대대 의무실", "title": "독감 예방접종 3중대 일정 변경", "content": "3중대 독감 예방접종을 11월 12일(목) 09시에서 11월 12일(목) 14시로 변경합니다. 오전 훈련 일정 조정에 따른 것입니다. 1중대와 2중대 일정은 변경 없습니다."},
    "N11": {"date": "11-09", "dept": "대대 군수과", "title": "생활관 난방기 점검", "content": "11월 11일(수)부터 13일(금)까지 생활관 난방기를 점검합니다. 점검 시간에는 생활관을 비워 두고, 귀중품은 개인 관물대에 보관합니다. 난방기 이상 소음이나 냄새가 있으면 점검 전까지 중대 행정반에 신고합니다."},
    "N12": {"date": "11-09", "dept": "대대 교육훈련과", "title": "AI 활용 교육 참가자 모집", "content": "대대 자체 AI 활용 교육 참가자를 모집합니다. 교육은 11월 23일(월)부터 4주간 매주 월요일 19시~21시에 사이버지식정보방에서 진행합니다. 중대별 5명 이내로 11월 16일(월)까지 중대 행정반에 신청합니다. 부대 자료를 외부 AI 서비스에 입력하지 않는 보안 수칙 교육이 포함됩니다."},
    "A1": {"date": "11-10", "dept": "대대 복지과", "title": "동계 체육대회 참가 신청", "content": "11월 27일(금) 대대 체육대회를 실시합니다. 종목은 축구, 족구, 줄다리기이며 중대별 참가 명단을 11월 18일(수)까지 복지과에 제출합니다. 우천 시 11월 30일(월)로 연기합니다. 문의: 복지과(내선 1503)."},
    "A2": {"date": "11-10", "dept": "대대 정보통신실", "title": "업무용 PC 보안 업데이트", "content": "11월 12일(목) 18시부터 20시까지 업무용 PC 보안 업데이트를 진행합니다. 업데이트 중에는 PC를 끄지 말고, 18시 전까지 작업 중인 문서를 저장합니다. 업데이트 후 오류가 있으면 정보통신실(내선 1401)에 신고합니다."}
}

def view_notice(notice_id):
    """가상 공지 원문을 편리하게 확인하는 함수"""
    if notice_id in NOTICES_DB:
        n = NOTICES_DB[notice_id]
        print(f"[{notice_id}] {n['title']} (작성: {n['date']} / {n['dept']})")
        print(f"본문: {n['content']}")
    else:
        print(f"존재하지 않는 공지 ID입니다: {notice_id}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 요약 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 요약 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 공지 데이터베이스(N01~N12, A1~A2) 준비 완료!")



---

## 🎯 미션 01. 공지 읽기의 어려움 파악하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **공지를 읽을 때 무엇을 자주 놓치는지 파악하고, AI 요약이 도움이 될 부분을 정합니다.** 가상 공지 12건(N01~N12)을 직접 읽고 핵심 정보 5종(기한, 대상, 장소, 준비물, 연락처)을 추출하여 수작업의 취약점을 분석하고 인간-AI 분업 기준을 수립합니다.

**이번 미션 결과물:** `공지 핵심 정보 체크표, 놓친 정보 기록, AI 활용 범위 정리` 완성

### 📑 실습 서식 및 대조 점검 양식
| 공지 번호 | 기한 | 대상 | 장소 | 준비물·제출물 | 연락처 | 놓친 정보 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| N01 | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| N02 | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| N03 | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ... | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

- **총 소요 시간**: (    )분
- **AI에게 위임할 과업 (단순 요약/추출)**: (                      )
- **사람이 직접 확인할 과업 (최종 검증/책임)**: (                      )

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 문제 정의·AI 적용 판단 — 공지에서 놓치기 쉬운 정보를 찾아내고 AI가 도울 부분과 사람이 확인할 부분을 구분했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 01. 공지 읽기의 어려움 파악하기] 프롬프트 및 실습 결과 기록
mission_01_prompt = """# [인간-AI 분업 기준 설계를 위한 사전 탐색 질의]
부대 행정 공지문에서 장병들이 일과 중 절대 놓쳐서는 안 되는 필수 5대 행정 정보를 정의하고,
줄글 형태 공지 확인 시 인간이 흔히 범하는 누락 실수 유형과 이를 보완하기 위한 AI 협업 방안을 제시해 줘.
"""

mission_01_result = """### 📋 1. 가상 공지 12건 핵심 행정 정보 체크표 (완성본)

| 공지 번호 | 기한 (마감일시) | 대상 | 장소 | 준비물·제출물 | 연락처 | 놓친 정보 (자가 점검) |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| **N01** | 11/05(목) 18:00 | 11월 정기휴가 희망 장병 | 중대 행정반 | 휴가 신청서 (분대장 제출) | 행정과 (내선 1101) | 정원 20% 초과시 추첨 조건 놓치기 쉬움 |
| **N02** | 11/10~12 (중대별 상이) | 전 장병 (미접종자 16일) | 의무실 앞 | 체온 확인 (발열시 사전고지) | 대대 의무실 | 접종 후 30분 대기 조건 |
| **N03** | 치수 11/04, 보급 11/06 13~17시 | 전 분대원 | 대대 군수과 | 분대원 치수표 제출 | 대대 군수과 | 교환은 보급 당일만 가능하다는 점 |
| **N04** | 11/08(일)까지 변경 | 업무용 계정 보유 전원 | 각 사무실 | 9자리 이상 신규 비밀번호 | 대대 정보통신실 | 모바일 업무자료 촬영 절대 금지 수칙 |
| **N05** | 11/06(금) -> 11/13(금) 연기 | 2중대 (1·3중대 변경없음) | 대대 강당 (실내) | 사격술 예비훈련 복장 | 교육훈련과 | 2중대만 해당된다는 대상 제한 |
| **N06** | 11/09(월)부터 시행 | 복지시설 이용 장병 | 체력단련실/사지방 | 이용 수칙 준수 | 대대 복지과 | 수요일 14~16시 소독 휴무 |
| **N07** | 11/20(금)까지 | 자기개발 희망 장병 | 인사과 제출 | 응시 확인서 / 수강 등록증 | 대대 인사과 | 결과가 12월 첫째 주 개별 통보라는 점 |
| **N08** | 11/06(금) 12:00 (연장) | 휴가 신청 장병 | 중대 행정반 | 휴가 신청서 | 행정과 (내선 1101) | N01의 마감일시 변경 사실 인지 필요 |
| **N09** | 상시 (동절기) | 전 장병 | 생활관 및 야외 | 방한장구 (2인1조) | 대대 의무실 | 생활관 내 개인 전열기 사용 금지 |
| **N10** | 11/12(목) 14:00 (시간변경) | 3중대 장병 | 의무실 앞 | 접종 전 발열 체크 | 대대 의무실 | 3중대만 오전->오후로 시간 변경됨 |
| **N11** | 11/11(수)~13(금) | 생활관 인원 | 생활관 | 귀중품 관물대 보관 | 중대 행정반 | 점검 시간 중 생활관 퇴실 의무 |
| **N12** | 11/16(월)까지 신청 | 중대별 5명 이내 | 사지방 | 신청서 제출 | 교육훈련과 | 보안 수칙 교육 필수 이수 조건 |

### ⏱️ 2. 소요 시간 측정 및 취약점 분석
- **12건 전체 수작업 독해 소요 시간**: 총 24분 (1건당 평균 2분 소요)
- **가장 놓치기 쉬웠던 정보 유형**: 
  1. 기한 변경 및 일정 연기 공지 (N08, N10): 이전 공지와 연결해 보지 않으면 놓침
  2. 대상자 예외 및 조건부 추첨 조항 (N01 20% 초과시 추첨, N05 2중대만 연기)

### 🤝 3. 인간-AI 역할 분업 정의서 (매트릭스)
- **AI 위임 과업 (단순 요약/추출)**: 긴 공지문에서 5대 항목(기한/대상/장소/준비물/연락처) 자동 추출, 날짜순 정렬 및 불릿포인트 표준 카드 형식화
- **인간 직접 확인 과업 (최종 검증/책임)**: 날짜·시간의 원문 1:1 대조, 변경 공지 반영 여부 교차 검증, 보안 위규 사항(실제 부대정보 입력 금지) 통제
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 01. 공지 읽기의 어려움 파악하기", mission_01_prompt, mission_01_result)



---

## 🎯 미션 02. 요약에 쓸 AI 도구 고르기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

군 행정 공지 요약에 가장 적합한 생성형 AI 도구를 선정하기 위해 2가지 이상의 공개 AI 도구를 실증적으로 비교·평가합니다. 동일한 공지 원문(N03)을 각각 입력해 요약 결과를 추출하고, 5대 평가 기준으로 분석하여 목적과 군 보안 수칙에 맞는 도구를 선택합니다.

**이번 미션 결과물:** `도구 비교표, 선택 이유` 완성

### 📑 실습 서식 및 대조 점검 양식
| 평가 기준 | 도구 A (예: ChatGPT) | 도구 B (예: Claude / Wrtn) | 비고 / 관찰 메모 |
| :--- | :--- | :--- | :--- |
| 1. 응답 속도 | (  )점 / 5점 | (  )점 / 5점 | |
| 2. 요약 분량 (적정성) | (  )점 / 5점 | (  )점 / 5점 | |
| 3. 날짜 표기 정확성 | (  )점 / 5점 | (  )점 / 5점 | |
| 4. 한국어 자연스러움 | (  )점 / 5점 | (  )점 / 5점 | |
| 5. 보안 준수 편의성 | (  )점 / 5점 | (  )점 / 5점 | |
| **총점** | **(  )점** | **(  )점** | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] AI 도구 선택 — 비교 기준을 세워 도구를 비교하고 목적과 보안 수칙에 맞는 도구를 골랐는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 02. 요약에 쓸 AI 도구 고르기] 프롬프트 및 실습 결과 기록
mission_02_prompt = """# [도구 벤치마킹 테스트 프롬프트 - 동일 공지 N03 대상]
아래 부대 공지사항을 읽고 분대원들이 알아야 할 핵심 내용을 3개 항목으로 요약해 줘.

[공지 N03 내용]:
동계 피복(방한 내피, 방한 장갑, 넥워머)을 11월 6일(금) 13시부터 17시까지 중대 순서대로 보급합니다. 분대장은 분대원 치수표를 11월 4일(수)까지 군수과에 제출합니다. 치수 교환은 보급 당일에만 가능하며, 이후에는 다음 달 정기 보급 때 교환합니다.
"""

mission_02_result = """### 📊 도구별 5대 기준 실증 비교 평가표

| 평가 기준 | 도구 A (ChatGPT-4o mini) | 도구 B (Claude 3.5 Sonnet) | 비고 / 관찰 메모 |
| :--- | :---: | :---: | :--- |
| **1. 응답 속도 (체감)** | 5점 (약 1.5초) | 4점 (약 3초) | 두 도구 모두 실무 활용에 충분히 신속함 |
| **2. 요약 분량 적정성** | 5점 (요청한 3줄 준수) | 5점 (깔끔한 불릿 형태 3줄) | 군더더기 없이 핵심만 축약함 |
| **3. 날짜 표기 정확성** | 5점 (11/4 치수제출, 11/6 보급) | 5점 (마감 및 보급 시간 정확 표기) | 날짜 왜곡이나 환각 없음 |
| **4. 한국어 가독성** | 4.5점 (군대 행정 어투 무난) | 5점 (자연스럽고 정돈된 표현) | Claude가 군사/공공 서식 표현에 약간 더 자연스러움 |
| **5. 보안 설정 편의성** | 4.5점 (임시 채팅/학습 제외 설정) | 4.5점 (계정 설정 내 데이터 학습 제외) | 두 도구 모두 입력 데이터 학습 제외 기능 지원 |
| **총점** | **23.0점 / 25점** | **23.5점 / 25점** | |

### 🏆 최종 주력 도구 선정 결과 및 근거
- **선정 도구**: **도구 A (ChatGPT)** (또는 무료 접근성이 뛰어난 공개 도구)
- **선정 이유**:
  1. 분대 공지 요약 실습 환경에서 계정 생성 및 모바일 앱 접근성이 가장 용이함.
  2. 날짜(11/4, 11/6) 및 마감 시간 추출에서 환각 없이 원문 팩트를 100% 보존함.
  3. 로그인 후 브라우저 환경에서 임시 채팅(Temporary Chat) 기능을 통해 데이터가 서버 학습에 사용되지 않도록 즉시 제어 가능하여 국방 보안 수칙 준수에 부합함.
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 02. 요약에 쓸 AI 도구 고르기", mission_02_prompt, mission_02_result)



---

## 🎯 미션 03. 첫 요약 프롬프트 작성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

기본적인 지시문으로 공지 1건(N01)을 요약하는 첫 프롬프트를 작성하고 AI의 요약 결과를 받아봅니다. AI가 생성한 요약문에서 필수 5대 행정 정보가 어떻게 반영되었는지 확인하고, 누락되거나 왜곡된 정보를 분석합니다.

**이번 미션 결과물:** `첫 3줄 요약 프롬프트 및 누락 정보 분석표` 완성

### 📑 실습 서식 및 대조 점검 양식
- **입력 프롬프트**: "다음 공지를 3줄로 요약해 줘."
- **필수 요소 식별**: 기한 포함 여부, 대상 포함 여부, 제출처 포함 여부
- **빠진 치명적 정보**: (                                            )

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 첫 요약 실행 — 공지 1건을 요약하고 빠진 필수 정보(기한·대상 등)를 찾았는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 03. 첫 요약 프롬프트 작성하기] 프롬프트 및 실습 결과 기록
mission_03_prompt = """다음 공지사항을 읽고 3줄로 요약해 줘.

[공지 내용]:
11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. 휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. 결과는 11월 9일(월) 중대 게시판에 게시합니다. 문의: 행정과 담당관(내선 1101).
"""

mission_03_result = """### 🤖 AI 기본 3줄 요약 생성 결과
1. 11월 정기 휴가 신청은 11월 5일(목) 18시까지 접수받습니다.
2. 신청서는 중대 행정반에서 작성하여 분대장에게 제출해야 합니다.
3. 관련 문의사항은 대대 행정과 담당관(내선 1101)에게 문의 바랍니다.

---
### 🔍 필수 요소 식별 및 누락 정보 분석표
- **기한 포함 여부**: [x] 포함 (11월 5일(목) 18시 명시됨)
- **대상 및 제출처 포함**: [x] 포함 (분대장에게 제출 명시됨)
- **문의처 포함 여부**: [x] 포함 (내선 1101 명시됨)
- **⚠️ 빠진 치명적 정보 (원문 대조 팩트체크)**:
  1. **[추첨 조건 누락]**: 원문의 "휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 추첨으로 조정"한다는 중요 심사 기준이 완전히 빠짐.
  2. **[결과 발표일 누락]**: "결과는 11월 9일(월) 중대 게시판에 게시"된다는 후속 일정이 누락됨.
- **분석 결론**: 단순 '3줄 요약' 프롬프트는 일반적인 안내는 잘 추리지만, 행정적 조건부 예외(추첨)와 후속 일정 같은 세부 필수 정보를 임의로 버리는 한계가 드러남.
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 03. 첫 요약 프롬프트 작성하기", mission_03_prompt, mission_03_result)



---

## 🎯 미션 04. 요약 결과 확인하고 바로잡기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

AI가 생성한 요약문을 원문과 문장 단위로 1:1 대조하여 오류, 왜곡, 환각(Hallucination) 현상을 식별합니다. 발견된 오류를 구체적으로 지적하는 교정 프롬프트를 작성하여 AI에게 재요청하고 무결점 요약본을 완성합니다.

**이번 미션 결과물:** `요약-원문 1:1 교차 대조표 및 교정 완료본` 완성

### 📑 실습 서식 및 대조 점검 양식
| 요약문 문장 | 원문 근거 문장 | 오류 / 왜곡 / 환각 여부 | 교정 내용 |
| :--- | :--- | :--- | :--- |
| 1행: | | [ ] 정상 / [ ] 오류 | |
| 2행: | | [ ] 정상 / [ ] 오류 | |
| 3행: | | [ ] 정상 / [ ] 오류 | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 대조 점검 — 요약과 원문을 1:1 대조해 오류나 빠진 부분을 찾아 고쳤는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 04. 요약 결과 확인하고 바로잡기] 프롬프트 및 실습 결과 기록
mission_04_prompt = """# [오류 교정 피드백 프롬프트]
앞서 작성해 준 요약문에서 다음 2가지 중요 행정 정보가 누락되었습니다.
1. '중대 정원의 20% 초과 시 선착순이 아닌 추첨 조정' 규칙
2. '결과 게시일: 11월 9일(월) 중대 게시판'

원문의 날짜와 숫자를 왜곡하지 말고, 위 2가지 정보를 반드시 포함하여 정확한 3개 항목으로 다시 요약해 줘.
"""

mission_04_result = """### 📋 1. 요약-원문 1:1 교차 대조표 (Fact-checking Sheet)

| 요약문 항목 | 원문 근거 문장 | 오류 / 누락 / 환각 여부 | 교정 조치 내용 |
| :--- | :--- | :---: | :--- |
| 1. 11월 5일(목) 18시까지 신청 접수 | 11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. | [정상] 일치 | 날짜·시간 원문과 100% 일치 |
| 2. 분대장에게 신청서 제출 | 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. | [정상] 일치 | 행정반 비치 서식 및 분대장 제출 유지 |
| (누락 항목 1) | 휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. | **[치명적 누락]** | 분대원 민원 방지를 위해 '20% 초과시 추첨' 항목 필수 삽입 |
| (누락 항목 2) | 결과는 11월 9일(월) 중대 게시판에 게시합니다. | **[누락]** | 발표일(11/9) 및 확인 장소(게시판) 추가 |

---
### ✅ 2. 피드백 후 재생성된 무결점 요약 결과
1. **[신청 기한 및 방법]**: 11월 5일(목) 18시까지 중대 행정반의 신청서를 작성하여 소속 분대장에게 제출.
2. **[선발 기준 및 결과]**: 중대 정원 20% 초과 시 추첨으로 조정하며, 결과는 11월 9일(월) 중대 게시판에 공지.
3. **[문의처]**: 대대 행정과 담당관 (내선 1101).
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 04. 요약 결과 확인하고 바로잡기", mission_04_prompt, mission_04_result)



---

## 🎯 미션 05. 역할과 조건으로 프롬프트 개선하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

AI에게 명확한 페르소나(분대장)를 부여하고 필수 항목 지정, 환각 금지('원문에 없는 내용은 지어내지 말 것') 등 엄격한 제약조건을 설정합니다. 개선된 프롬프트를 공지 3건(N01~N03)에 적용하여 기본 요약 대비 누락과 환각이 어떻게 줄어드는지 전후 비교표로 입증합니다.

**이번 미션 결과물:** `역할·제약조건 개선 프롬프트 및 전후 비교표` 완성

### 📑 실습 서식 및 대조 점검 양식
| 공지 번호 | 기본 3줄 요약 문제점 | 역할·조건 지정 후 개선점 | 환각 방지 확인 |
| :--- | :--- | :--- | :--- |
| N01 (휴가) | | | [ ] 확인 |
| N02 (접종) | | | [ ] 확인 |
| N03 (피복) | | | [ ] 확인 |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 프롬프트 개선 — 역할·조건을 넣어 프롬프트를 개선하고 전후 차이를 비교했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 05. 역할과 조건으로 프롬프트 개선하기] 프롬프트 및 실습 결과 기록
mission_05_prompt = """너는 분대원들에게 공지를 알기 쉽게 전달하는 분대장이다.
아래 공지사항을 요약하되 반드시 다음 조건을 지켜라.

[제약조건]:
1. [기한], [대상], [준비물], [문의처] 4개 항목으로 구분하여 정리할 것.
2. 원문에 없는 내용은 절대 지어내지 말고 '원문 미표기'로 적을 것.
3. 날짜와 시간은 원문 표현 그대로 정확히 인용할 것.

[공지 내용]:
[공지 N01, N02, N03 순차 적용]
"""

mission_05_result = """### 📋 공지 3건(N01~N03) 역할·조건 지정 후 산출물 및 비교표

#### [N01 요약 결과]
- **기한**: 11월 5일(목) 18시까지 (결과 게시: 11월 9일(월))
- **대상**: 11월 정기 휴가 희망 장병 (중대 정원 20% 초과 시 추첨)
- **준비물**: 중대 행정반 비치 휴가 신청서
- **문의처**: 대대 행정과 담당관 (내선 1101)

#### [N02 요약 결과]
- **기한**: 11월 10일(화)~12일(목) (1중대 10일 09시, 2중대 11일 09시, 3중대 12일 09시 / 미접종자 16일)
- **대상**: 전 장병 (접종 당일 발열 인원은 의무병 사전 통보)
- **준비물**: 접종 후 30분간 의무실 대기 준수
- **문의처**: 대대 의무실

#### [N03 요약 결과]
- **기한**: 치수표 제출 11월 4일(수)까지 / 피복 보급 11월 6일(금) 13:00~17:00
- **대상**: 전 분대원 (분대장이 치수표 취합)
- **준비물**: 분대원 치수표 (교환은 보급 당일만 가능)
- **문의처**: 대대 군수과

---
### 📊 역할·조건 지정 전후 품질 비교표
| 공지 번호 | 기본 3줄 요약 문제점 | 역할·조건 지정 후 개선점 | 환각 방지 확인 |
| :---: | :--- | :--- | :---: |
| **N01 (휴가)** | 추첨 기준(20%) 및 결과 게시일 누락 | 4대 슬롯 구조화로 마감일시 및 조건 완벽 포함 | [x] 완벽 확인 |
| **N02 (접종)** | 중대별 집합 시간 뭉뚱그려 표시 | 중대별 날짜·시간(10/11/12일 09시) 정확 인용 | [x] 완벽 확인 |
| **N03 (피복)** | 치수표 제출일(11/4)이 누락될 위험 | 치수표 제출일과 보급일시가 명확히 분리 명시됨 | [x] 완벽 확인 |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 05. 역할과 조건으로 프롬프트 개선하기", mission_05_prompt, mission_05_result)



---

## 🎯 미션 06. 한눈에 보이는 요약 형식 정하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

모바일 메신저나 상황판에 즉시 게시할 수 있는 가독성 높은 표준 카드 템플릿 슬롯(무엇을, 언제까지, 누가, 어디서, 문의)을 정의합니다. Few-shot(예시 제시) 기법을 활용하여 AI가 지정된 규격과 이모지 포맷을 엄격히 준수하도록 훈련합니다.

**이번 미션 결과물:** `모바일 표준 요약 카드 템플릿 및 Few-shot 프롬프트` 완성

### 📑 실습 서식 및 대조 점검 양식
너는 분대장이다. 아래 [예시]의 형식과 스타일을 그대로 따라서 [새 공지]를 요약 카드 형태로 작성하라.
[예시 카드] -> [새 공지 입력]

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 형식 통일 — 요약 형식을 정하고 Few-shot(예시)으로 형식을 통일했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 06. 한눈에 보이는 요약 형식 정하기] 프롬프트 및 실습 결과 기록
mission_06_prompt = """너는 분대장이다. 아래 [예시]의 형식과 스타일을 그대로 따라서 [새 공지]를 모바일 요약 카드 형태로 작성하라.

---
[예시]
📌 [공지] 동계 피복 보급 일정 안내
• 무엇을: 동계 피복(방한 내피·장갑·넥워머) 수령 및 치수표 제출
• 언제까지: 치수표 제출 11월 4일(수)까지 / 보급 11월 6일(금) 13:00~17:00
• 누가: 전 분대원 (분대장이 치수표 취합 제출)
• 어디서: 대대 군수과
• 문의: 군수과 (당일 교환만 가능)
---

[새 공지]:
[N01, N02, N04 공지 입력]
"""

mission_06_result = """### 📱 Few-shot 기반 모바일 표준 요약 카드 3건 완성본

#### 📌 [카드 1] 11월 정기 휴가 신청 접수 안내 (N01)
• **무엇을**: 11월 정기 휴가 신청서 작성 및 제출 (20% 초과시 추첨)
• **언제까지**: 11월 5일(목) 18:00까지 (결과 발표: 11월 9일 중대 게시판)
• **누가**: 11월 정기 휴가 희망 전 장병
• **어디서**: 중대 행정반 (서식 작성 후 분대장에게 제출)
• **문의**: 대대 행정과 담당관 (내선 1101)

#### 📌 [카드 2] 독감 예방접종 일정 안내 (N02)
• **무엇을**: 독감 예방접종 실시 (접종 후 30분 대기 필수, 발열시 사전 신고)
• **언제까지**: 11월 10일(화)~12일(목) 09:00 (1중대 10일, 2중대 11일, 3중대 12일 / 미접종자 16일)
• **누가**: 대대 전 장병
• **어디서**: 대대 의무실 앞 집합
• **문의**: 대대 의무실

#### 📌 [카드 3] 사이버 보안 점검 및 비밀번호 변경 (N04)
• **무엇을**: 업무용 계정 비밀번호 변경 (영문·숫자·특수문자 포함 9자리 이상)
• **언제까지**: 11월 8일(일)까지 (보안 점검 기간: 11/9~13)
• **누가**: 업무용 계정 보유 전 인원
• **어디서**: 각 행정 사무실 PC
• **문의**: 대대 정보통신실 (⚠️ 개인 휴대전화 업무자료 촬영 금지)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 06. 한눈에 보이는 요약 형식 정하기", mission_06_prompt, mission_06_result)



---

## 🎯 미션 07. 여러 공지를 한 번에 정리하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

가상 공지 12건 전체를 한꺼번에 다룰 때 발생하는 컨텍스트 누락을 방지하기 위해, 공지를 청크(Chunk, 4건씩 3개 묶음)로 나누어 마감 기한을 정밀 추출합니다. 추출된 기한을 시간 순서대로 재정렬하여 분대원들이 따라야 할 주간 할 일(To-Do) 목록을 도출합니다.

**이번 미션 결과물:** `기한순(D-day) 공지 정렬표 및 주간 분대 할 일(To-Do) 목록` 완성

### 📑 실습 서식 및 대조 점검 양식
| 마감 일시 | 공지 번호 | 공지 제목 | 주요 조치 사항 |
| :--- | :--- | :--- | :--- |
| 11월 4일(수) | N03 | 동계 피복 치수표 제출 | 군수과 제출 |
| ... | | | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 다중 공지 처리 — 12건 공지에서 기한을 추출해 기한순 목록을 만들었는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 07. 여러 공지를 한 번에 정리하기] 프롬프트 및 실습 결과 기록
mission_07_prompt = """다음은 한빛대대의 가상 공지 12건(N01~N12)이다.
장병들이 마감 기한을 놓치지 않도록, 모든 공지에서 기한(날짜와 시간)을 빠짐없이 추출하여
가장 빠른 날짜부터 순서대로(D-day 기한순) 정렬된 일정표와 분대 주간 To-Do 목록을 작성하라.

[공지 N01~N12 데이터 입력]
"""

mission_07_result = """### 🗓️ 1. 기한순(D-day) 전체 공지 정렬표 (마감 골드 데이터셋 대조)

| 마감 일시 | 공지 번호 | 공지 제목 | 주요 조치 및 담당 행동 |
| :---: | :---: | :--- | :--- |
| **11월 4일(수)** | **N03** | 동계 피복 치수표 제출 | 분대장이 분대원 치수표 취합 후 군수과 제출 |
| **11월 5일(목) 18:00** | **N01** | 11월 정기 휴가 신청 접수 | 중대 행정반 서식 작성 후 분대장 제출 |
| **11월 6일(금) 13~17시** | **N03** | 동계 피복 보급 수령 | 중대 순서대로 수령 및 현장 치수 확인 (당일 교환) |
| **11월 8일(일)** | **N04** | 업무 계정 비밀번호 변경 | 9자리 이상(영문·숫자·특수문자) 변경 완료 |
| **11월 9일(월)부터** | **N06** | 복지시설 동절기 운영시간 조정 | 체력단련실(06~21시), 사지방(18~21:30) 적용 |
| **11월 10일(화)~12일(목)** | **N02** | 독감 예방접종 실시 | 1중대 10일 09시, 2중대 11일 09시, 3중대 12일 09시 의무실 앞 집합 |
| **11월 11일(수)~13일(금)** | **N11** | 생활관 난방기 점검 | 점검 시간 생활관 비우기, 귀중품 관물대 보관 |
| **11월 16일(월)** | **N12** | AI 활용 교육 참가 신청 마감 | 중대 행정반 접수 (중대별 5명 선발) |
| **11월 20일(금)** | **N07** | 자기개발 지원비 신청 마감 | 응시 확인서/수강증 첨부하여 인사과 제출 |

---
### ✅ 2. 이번 주 분대 주간 할 일 (To-Do) 목록
- [ ] **11/04(수) 마감**: [피복] 분대원 동계 피복 치수표 취합 후 군수과 제출 완료
- [ ] **11/05(목) 18:00 마감**: [휴가] 11월 정기 휴가 신청서 행정반 비치 서식 분대장 제출
- [ ] **11/06(금) 13:00~17:00**: [피복] 동계 피복 수령 및 불량/치수 교환 (당일 교환 필수)
- [ ] **11/08(일) 자정까지**: [보안] 전원 업무용 계정 비밀번호 9자리 이상 변경 (촬영 금지)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 07. 여러 공지를 한 번에 정리하기", mission_07_prompt, mission_07_result)



---

## 🎯 미션 08. 변경 공지와 빠진 정보 점검하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

기존 공지(N01 휴가, N02 접종, N05 사격)가 나중에 발행된 변경 공지(N08 휴가연장, N10 독감시간변경)로 인해 수정된 이력을 추적합니다. 변경 사항을 누락 없이 반영하고 구버전 마감일이 잔존하지 않도록 최종 무결성을 점검합니다.

**이번 미션 결과물:** `공지 변경 이력 대조표 및 최종 무결성 점검표` 완성

### 📑 실습 서식 및 대조 점검 양식
| 변경 공지 | 원본 공지 | 기존 내용 | 변경된 최신 내용 | 변경 사유 |
| :--- | :--- | :--- | :--- | :--- |
| N08 | N01 | 11월 5일(목) 18시 | 11월 6일(금) 12시 | 사격 훈련 일정 변경 |
| N10 | N02 | 3중대 12일 09시 | 3중대 12일 14시 | 훈련 일정 조정 |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 변경 반영·재점검 — 변경 공지를 찾아 내용을 고치고 빠진 정보를 보완했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 08. 변경 공지와 빠진 정보 점검하기] 프롬프트 및 실습 결과 기록
mission_08_prompt = """부대 공지 중에는 이전 공지의 일정을 변경하거나 연장하는 '정정 공지'가 존재한다.
N01~N12 공지를 상호 대조하여, 기존 내용을 수정하는 변경 공지들을 모두 식별하고
변경 전후의 차이점과 최신 수정 사항이 반영된 정정 To-Do 목록을 작성해 줘.
"""

mission_08_result = """### 📋 1. 공지 변경 이력 대조표 (Change Log)

| 변경 공지 | 원본 공지 | 원본 내용 (이전 기준) | 변경된 최신 내용 (최종 확정) | 변경 사유 및 영향 |
| :---: | :---: | :--- | :--- | :--- |
| **N08** (11/05발행) | **N01** (정기휴가) | 신청 마감: 11월 5일(목) 18:00 | **신청 마감: 11월 6일(금) 12:00까지 연장** | 사격 훈련 일정 변경에 따른 연장 조치 |
| **N10** (11/06발행) | **N02** (독감접종) | 3중대 접종: 11월 12일(목) 09:00 | **3중대 접종: 11월 12일(목) 14:00로 변경** | 오전 훈련 일정 조정 (1·2중대는 변동 없음) |
| **N05** (11/04발행) | 자체 변경 | 2중대 사격: 11월 6일(금) | **2중대 사격: 11월 13일(금)로 연기** | 기상 악화 예보로 6일 강당 실내교육 대체 |

---
### 🗓️ 2. 정정 반영된 최신 주간 To-Do 목록
- [ ] **11/04(수)**: [피복] 치수표 군수과 제출 마감
- [ ] **11/06(금) 12:00까지 [⚠️마감 연장 정정]**: [휴가] 정기 휴가 신청서 최종 마감 (N08 반영, 기존 11/5 18시 폐기)
- [ ] **11/06(금) 13:00~17:00**: [피복] 동계 피복 수령 (당일 치수교환 필수)
- [ ] **11/08(일)**: [보안] 업무 계정 비밀번호 9자리 이상 변경
- [ ] **11/12(목) 14:00 [⚠️시간 변경 정정]**: [의무] 3중대 독감 예방접종 (N10 반영, 오전 09시 집합 아님)

---
### 🔍 3. 최종 무결성 점검표
- [x] 이전 마감일(11/5 18시)이 To-Do 목록에서 완전히 삭제되고 연장된 11/6 12시로 대체되었는가? (PASS)
- [x] 3중대 접종 시간이 오전 09시에서 오후 14시로 올바르게 수정되었는가? (PASS)
- [x] 변경되지 않은 1중대(10일 09시), 2중대(11일 09시) 일정이 유지되었는가? (PASS)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 08. 변경 공지와 빠진 정보 점검하기", mission_08_prompt, mission_08_result)



---

## 🎯 미션 09. 나만의 공지 요약 프롬프트 만들기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

지금까지 습득한 역할(Persona), 제약조건(Constraints), Few-shot 템플릿, 변경 추적 지침을 집대성하여 재사용 가능한 '표준 공지 요약 마스터 프롬프트'를 구축합니다. 이 프롬프트를 처음 보는 신규 공지 2건(A1 체육대회, A2 PC 보안)에 수정 없이 적용하여 범용성을 검증합니다.

**이번 미션 결과물:** `표준 공지 요약 마스터 프롬프트 템플릿 및 3줄 사용 매뉴얼` 완성

### 📑 실습 서식 및 대조 점검 양식
[나만의 공지 요약 마스터 프롬프트 틀]
# 역할 / 제약조건 / 출력 서식 / 신규 공지 입력 슬롯

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 나만의 활용법 — 재사용 가능한 프롬프트 틀을 만들고 새 공지에 적용해 확인했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 09. 나만의 공지 요약 프롬프트 만들기] 프롬프트 및 실습 결과 기록
mission_09_prompt = """# ============================================================================
# [마스터 프롬프트 템플릿 검증 질의]
# 아래 표준 마스터 프롬프트 틀에 신규 공지 A1, A2를 각각 대입하여 요약 카드를 생성하라.
# ============================================================================

[마스터 프롬프트 원문]:
너는 부대 공지를 빠르고 정확하게 분대원들에게 전파하는 '분대장 AI 비서'이다.
제공되는 [공지사항 원문]을 바탕으로 아래 [지침]과 [출력 서식]을 엄격히 준수하여 요약 카드를 작성하라.

[엄격 준수 지침]:
1. 필수 5대 행정 항목(무엇을, 언제까지, 누가, 어디서, 문의)을 반드시 포함할 것.
2. 날짜, 시간, 숫자, 내선번호는 원문 그대로 표기할 것. 원문에 없는 내용은 '원문 미표기'로 적고 절대 상상하지 말 것.
3. 기한 변경이나 주의사항이 있으면 [⚠️주의] 태그를 붙여 강조할 것.

[출력 서식]:
📌 [공지] <제목>
• 무엇을: <주요 행동 및 내용>
• 언제까지: <마감 날짜 및 시간>
• 누가: <대상 인원>
• 어디서: <장소>
• 문의: <담당 부서 및 내선번호>

[신규 공지 A1, A2 순차 대입]
"""

mission_09_result = """### 🛠️ 1. 완성된 표준 공지 요약 마스터 프롬프트 템플릿
```text
너는 부대 공지를 빠르고 정확하게 분대원들에게 전파하는 '분대장 AI 비서'이다.
제공되는 [공지사항 원문]을 바탕으로 아래 [지침]과 [출력 서식]을 엄격히 준수하여 요약 카드를 작성하라.

[엄격 준수 지침]:
1. 필수 5대 행정 항목(무엇을, 언제까지, 누가, 어디서, 문의)을 반드시 포함할 것.
2. 날짜, 시간, 숫자, 내선번호는 원문 그대로 표기할 것. 원문에 없는 내용은 '원문 미표기'로 적고 절대 상상하지 말 것.
3. 기한 변경이나 주의사항이 있으면 [⚠️주의] 태그를 붙여 강조할 것.

[출력 서식]:
📌 [공지] <제목>
• 무엇을: <주요 행동 및 내용>
• 언제까지: <마감 날짜 및 시간>
• 누가: <대상 인원>
• 어디서: <장소>
• 문의: <담당 부서 및 내선번호>
```

---
### 🧪 2. 신규 공지(A1, A2) 무수정 적용 시험 결과

#### 📌 [공지] 동계 체육대회 참가 신청 (A1 적용본)
• **무엇을**: 대대 체육대회 종목(축구, 족구, 줄다리기) 중대별 참가 명단 제출
• **언제까지**: 명단 제출 11월 18일(수)까지 (대회 일시: 11월 27일(금), 우천시 11/30 연기)
• **누가**: 대대 전 중대 장병
• **어디서**: 원문 미표기 (우천 연기 규정 포함)
• **문의**: 복지과 (내선 1503)

#### 📌 [공지] 업무용 PC 보안 업데이트 (A2 적용본)
• **무엇을**: 업무용 PC 보안 업데이트 진행 (PC 끄지 말 것, 18시 전 문서 저장 완료)
• **언제까지**: 11월 12일(목) 18:00~20:00
• **누가**: 업무용 PC 사용자 전원
• **어디서**: 각 사무실 PC
• **문의**: 정보통신실 (내선 1401) [⚠️주의: 업데이트 오류 발생 시 즉시 신고]

---
### 📖 3. 비전문가를 위한 초간단 3줄 사용 매뉴얼
1. **[복사하기]**: 위 마스터 프롬프트 템플릿 전체를 복사하여 생성형 AI 대화창에 붙여넣습니다.
2. **[원문 넣기]**: 맨 아래 `[공지 원문]` 자리에 새로 하달된 가상 공지 텍스트를 붙여넣고 전송합니다.
3. **[원문 대조]**: 출력된 카드의 날짜와 시간이 원문과 일치하는지 1분간 대조 확인 후 분대원에게 전파합니다.
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 09. 나만의 공지 요약 프롬프트 만들기", mission_09_prompt, mission_09_result)



---

## 🎯 미션 10. 분대 공지 요약 카드 완성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

프로젝트의 최종 단계로, 이번 주 공지 전체를 한눈에 볼 수 있는 '주간 분대 공지 요약 카드' 패키지를 완성합니다. 모바일 가독성, 변경 공지 정정 완료, 기한순 To-Do 목록, 그리고 AI 산출물 면책 및 공식 확인 필수 안내문(Disclaimer)을 포함하여 배포 가능한 최종본을 산출합니다.

**이번 미션 결과물:** `최종 주간 분대 공지 요약 카드 배포 패키지 (미션10, 최종 필수)` 완성

### 📑 실습 서식 및 대조 점검 양식
=================================================
📢 [한빛대대 1중대 2분대 주간 공지 브리핑 카드]
=================================================
[공지 요약 카드들]
-------------------------------------------------
🗓️ 이번 주 분대 주간 할 일 (To-Do)
-------------------------------------------------
※ 원문 확인 필수 면책 안내문
=================================================

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 결과물 완성 — 요약 카드가 정확하고 읽기 쉬우며 원문 확인 안내를 포함했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 10. 분대 공지 요약 카드 완성하기] 프롬프트 및 실습 결과 기록
mission_10_prompt = """# [최종 주간 브리핑 카드 생성 지시]
이번 주 한빛대대 가상 공지 중 분대원들이 필수적으로 숙지해야 할 핵심 공지 3건(휴가 연장, 피복 보급, 보안 점검)과
기한순 To-Do 목록, 그리고 AI 요약본의 한계를 명시하는 공식 확인 면책 문구를 결합하여
모바일 단체방에 즉시 공유할 수 있는 [주간 분대 공지 브리핑 카드 패키지]를 완성하라.
"""

mission_10_result = """### 📢 최종 주간 분대 공지 요약 카드 배포 패키지 (최종본)

```text
=================================================
📢 [한빛대대 1중대 2분대 주간 공지 브리핑 카드]
기준일자: 2026. 11. 05. (목) | 작성자: 2분대장
=================================================

📌 [공지 1] 11월 정기 휴가 신청 [⚠️ 마감 기한 연장 / 원문 확인 필수: N01, N08]
• 무엇을: 휴가 신청서 작성 및 분대장 제출 (중대 정원 20% 초과시 추첨)
• 언제까지: 11월 6일(금) 12:00까지 [기한 연장됨]
• 누가: 11월 정기 휴가 희망 장병
• 어디서: 중대 행정반
• 문의: 대대 행정과 담당관 (내선 1101)

📌 [공지 2] 동계 피복 보급 및 치수표 제출 [N03]
• 무엇을: 피복 치수표 제출 및 당일 보급 수령 (방한 내피, 장갑, 넥워머)
• 언제까지: 치수표 제출 11/4(수)까지 / 보급 11/6(금) 13:00~17:00
• 누가: 전 분대원 (분대장이 취합 제출)
• 어디서: 대대 군수과
• 주의: 피복 치수 교환은 보급 당일에만 가능 (이후 교환 불가)

📌 [공지 3] 사이버 보안 점검 및 계정 비밀번호 변경 [⚠️ 원문 확인 필수: N04]
• 무엇을: 업무용 계정 비밀번호 변경 (영문·숫자·특수문자 포함 9자리 이상)
• 언제까지: 11월 8일(일)까지 (보안 점검 기간: 11/9~13)
• 누가: 업무용 계정 보유 전 인원
• 주의: 개인 휴대전화로 부대 업무 자료 촬영 및 전송 절대 금지

-------------------------------------------------
🗓️ 이번 주 분대 주간 할 일 (To-Do)
[v] 11/04(수): 동계 피복 치수표 군수과 제출 완료
[ ] 11/06(금) 12:00: [연장마감] 11월 정기 휴가 신청서 최종 마감
[ ] 11/06(금) 13:00~17:00: 동계 피복 수령 및 사이즈 확인 (당일 교환)
[ ] 11/08(일): 업무 계정 비밀번호 9자리 이상 변경 완료

-------------------------------------------------
⚠️ [공식 확인 및 면책 안내문]
본 요약 카드는 생성형 AI 기반 업무 보조 자료이므로, 중요한 세부 일정 및 결격 사유는 반드시 공지 원문 및 담당 행정 부서(지휘계통)에 최종 확인하시기 바랍니다.
=================================================
```

---
### 📋 수신자 1분 인지 사용성 평가표
- [x] **신속성**: 분대원이 1분 이내에 본인의 마감일시와 행동 지침을 즉각 인지할 수 있는가? (**적합**)
- [x] **정확성**: N08의 휴가 신청 연장(11/6 12시)이 반영되어 혼선이 방지되었는가? (**적합**)
- [x] **보안성**: 개인정보나 비인가 군사 데이터 없이 가상 자료 원칙을 완벽히 지켰는가? (**적합**)
- [x] **신뢰성**: 하단에 AI 보조 자료 면책 및 원문 확인 안내문이 명확히 표기되었는가? (**적합**)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 10. 분대 공지 요약 카드 완성하기", mission_10_prompt, mission_10_result)

